# **PROYECTO FINAL**

Curso: IA Generativa

Integrantes:
*   Marcelo Tamayo Macedo
*   Cesar Perez Palomino



1. Instalación de librerias para todo el pipeline RAG

In [23]:
!pip install -qU langchain langchain-community langchain-huggingface
!pip install -qU faiss-cpu sentence-transformers rank_bm25 wikipedia
!pip install -qU rouge-score nltk
!pip install -qU transformers accelerate bitsandbytes

2. Carga del Dataset (Elegimos la temática de Normativa y Leyes del Perú)

In [24]:
import wikipedia

wikipedia.set_lang("es")

# Temas para asegurar un dataset real de leyes peruanas con > 100 chunks
temas_leyes = [
    "Constitución Política del Perú",
    "Código Civil del Perú",
    "Código Penal del Perú",
    "Congreso de la República del Perú",
    "Poder Judicial del Perú"
]

texto_dataset = ""
print("Descargando dataset de Wikipedia...")

for tema in temas_leyes:
    try:
        pagina = wikipedia.page(tema)
        # Añadimos marcadores para que el RAG sepa de qué documento proviene
        texto_dataset += f"\n\n[DOC_TITLE: {pagina.title}]\n\n{pagina.content}\n"
    except Exception as e:
        print(f"Error con {tema}: {e}")

print(f"Dataset descargado. Longitud total: {len(texto_dataset)} caracteres.")

Descargando dataset de Wikipedia...
Dataset descargado. Longitud total: 74596 caracteres.


3. Chunking + Overlap
* Chunks grandes (>1000 tokens): Proveen más contexto al LLM, pero diluyen la
información específica, aumentan el costo computacional/tiempo y pueden exceder la ventana de contexto del LLM.
* Chunks pequeños (<200 tokens): Capturan significados muy precisos (mejor para FAISS), pero pueden perder el contexto general (ej. un pronombre él sin saber a quién refiere).
* Decisión: Usamos 500 caracteres con 100 de overlap. Es un balance ideal para leyes, donde los artículos suelen ser cortos, y el overlap evita cortar sentencias a la mitad.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks_raw = text_splitter.split_text(texto_dataset)

# Creamos una lista de diccionarios para mantener un ID/Index por cada chunk
chunks = [{"id": i, "text": chunk} for i, chunk in enumerate(chunks_raw)]

print(f"Total de chunks generados: {len(chunks)} (Cumple el mínimo de 100 chunks).")

Total de chunks generados: 242 (Cumple el mínimo de 100 chunks).


4, Embeddings

In [26]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Usamos un modelo multilingüe eficiente para español
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

textos_chunks = [c["text"] for c in chunks]
print("Generando embeddings...")
# Convertimos a numpy array de tipo float32 (requerido por FAISS)
embeddings = embedder.encode(textos_chunks, convert_to_numpy=True).astype('float32')
dim_embedding = embeddings.shape[1]

print(f"Embeddings generados. Dimensión: {dim_embedding}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando embeddings...
Embeddings generados. Dimensión: 384


5. FAISS HNSW + Hybrid MN25
* HNSW es un algoritmo de búsqueda aproximada de vecinos más cercanos (ANN).
* Crea un grafo con múltiples capas jerárquicas (small world networks).
* Permite búsquedas logarítmicas ultra rápidas O(log N) saltando entre nodos en las capas superiores y refinando en las inferiores.
* Es superior a IndexFlatL2 (búsqueda exhaustiva) en bases de datos masivas.

In [27]:
import faiss
from rank_bm25 import BM25Okapi


# Configuramos FAISS HNSW
M = 32 # Número de conexiones en el grafo por nodo
efSearch = 64 # Profundidad de búsqueda (tradeoff velocidad/precisión)
index_hnsw = faiss.IndexHNSWFlat(dim_embedding, M)
index_hnsw.hnsw.efSearch = efSearch

# Indexamos los vectores en FAISS
index_hnsw.add(embeddings)
print("Índice FAISS HNSW creado e inicializado.")

# Configuramos BM25 para búsqueda léxica (palabras exactas)
tokenized_corpus = [doc.split(" ") for doc in textos_chunks]
bm25 = BM25Okapi(tokenized_corpus)
print("Índice léxico BM25 creado.")

Índice FAISS HNSW creado e inicializado.
Índice léxico BM25 creado.


6. Retrieval (Búsqueda Híbrida)

In [28]:
def hybrid_search(query, top_k=10):
    # 1. FAISS Search (Semántica)
    query_vector = embedder.encode([query], convert_to_numpy=True).astype('float32')
    distancias_faiss, indices_faiss = index_hnsw.search(query_vector, top_k)

    # 2. BM25 Search (Léxica)
    tokenized_query = query.split(" ")
    scores_bm25 = bm25.get_scores(tokenized_query)
    indices_bm25 = np.argsort(scores_bm25)[::-1][:top_k]

    # Combinación (Unión simple de índices únicos)
    indices_hibridos = list(set(indices_faiss[0].tolist() + indices_bm25.tolist()))

    documentos_recuperados = [chunks[i] for i in indices_hibridos]
    return documentos_recuperados

print("Función de Retrieval Híbrido lista.")

Función de Retrieval Híbrido lista.


7. Reranker (Cross-Encoder)
* A diferencia de los bi-encoders (FAISS) que comparan vectores pre-calculados por separado, un cross-encoder procesa la Query y el Documento juntos en la misma red neuronal.
* Esto captura relaciones semánticas complejas y cruzadas, siendo mucho más preciso, aunque más lento. Por eso se usa solo para reordenar los top-K resultados rápidos de FAISS/BM25.


In [29]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

def rerank_documents(query, documentos, top_n=3):
    # Preparamos pares (Query, Documento)
    pares = [[query, doc["text"]] for doc in documentos]

    # El cross-encoder nos da un score de relevancia
    scores = reranker_model.predict(pares)

    # Añadimos los scores a los documentos y ordenamos de mayor a menor
    for i, doc in enumerate(documentos):
        doc["score"] = scores[i]

    doc_ordenados = sorted(documentos, key=lambda x: x["score"], reverse=True)

    # Devolvemos solo los N más relevantes
    return doc_ordenados[:top_n]

print("Función de Reranker lista.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Función de Reranker lista.


8 - 9. LLM QWEN + Citation Per Sentence

In [30]:
import torch
from transformers import pipeline

# Cargamos un modelo Qwen pequeño apto para la RAM de Colab
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto"
)

def generate_answer_with_citations(query, docs_recuperados):
    # Preparamos el contexto numerando los documentos para la citación
    context_str = ""
    for i, doc in enumerate(docs_recuperados):
        context_str += f"[{i+1}] {doc['text']}\n\n"

    # PROMPT DE CITATION PER SENTENCE
    sys_prompt = (
        "Eres un asistente legal experto. Responde a la pregunta basándote estrictamente en el contexto proveído. "
        "REQUISITO OBLIGATORIO: Por CADA oración que generes, debes incluir la cita del documento exacto de donde "
        "sacaste la información, usando el formato [Doc X] al final de la oración. "
        "Si el contexto no tiene la respuesta, di 'No se encontró información'."
    )

    user_prompt = f"Contexto:\n{context_str}\n\nPregunta: {query}\nRespuesta:"

    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt},
    ]

    outputs = pipe(messages, max_new_tokens=256, temperature=0.1, max_length=None)
    return outputs[0]["generated_text"][-1]["content"], context_str

print("Pipeline LLM Qwen listo.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Pipeline LLM Qwen listo.


10. HALLUCINATION GUARD

* Implementamos un guardarraíl simple basado en solapamiento de tokens (Token overlap).
* Si la respuesta usa muchas palabras que no están en el contexto, hay riesgo de alucinación.

In [31]:
def hallucination_guard(respuesta, contexto, umbral=0.4):
    import re
    # Limpiamos y tokenizamos la respuesta y el contexto
    tokens_resp = set(re.findall(r'\w+', respuesta.lower()))
    tokens_ctx = set(re.findall(r'\w+', contexto.lower()))

    # Quitamos stopwords simples y números para no penalizar el formato [Doc X]
    tokens_resp = {t for t in tokens_resp if not t.isnumeric() and len(t) > 3}

    # Calculamos cuántas palabras significativas de la respuesta ESTÁN en el contexto
    if len(tokens_resp) == 0: return False, 1.0

    overlap = len(tokens_resp.intersection(tokens_ctx)) / len(tokens_resp)

    hay_alucinacion = overlap < umbral

    return hay_alucinacion, overlap

print("Guardarraíl contra alucinaciones listo.")

Guardarraíl contra alucinaciones listo.


11. GROUNDING EVALUATION (Score 0-1)

* Grounding evalúa qué tan anclada (fundamentada) está la respuesta en los documentos recuperados.
* Usaremos el modelo cross-encoder (que también usamos para reranking) como un modelo NLI (Natural Language Inference) improvisado para verificar si el contexto implica la respuesta.

In [32]:
def evaluate_grounding(respuesta, contexto):
    # Truncamos strings muy largos para evitar desbordar el modelo
    ctx_trunc = contexto[:1500]
    ans_trunc = respuesta[:500]

    # Evaluamos qué tanta similitud/implicación hay entre el texto base y la respuesta
    score_crudo = reranker_model.predict([ctx_trunc, ans_trunc])

    # Aplicamos una función sigmoide para asegurar que el grounding score esté estrictamente entre 0 y 1
    grounding_score = 1 / (1 + np.exp(-score_crudo))
    return float(grounding_score)

print("Evaluador de Grounding listo.")

Evaluador de Grounding listo.


12. BLEU / ROUGE + Ejecucion del Pipeline

In [33]:
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Descargamos tokenizadores de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

def evaluar_metricas(respuesta_generada, respuesta_referencia):
    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores_rouge = scorer.score(respuesta_referencia, respuesta_generada)

    # BLEU
    ref_tokens = nltk.word_tokenize(respuesta_referencia.lower())
    gen_tokens = nltk.word_tokenize(respuesta_generada.lower())
    chencherry = SmoothingFunction()
    score_bleu = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=chencherry.method1)

    return scores_rouge, score_bleu

# PIPELINE RAG COMPLETO (Demo)

pregunta_prueba = "¿Qué es el Poder Judicial?"
respuesta_referencia_esperada = "El Poder Judicial es el órgano encargado de administrar justicia en el país."

print(f"--- QUERY: {pregunta_prueba} ---")

# 1. Recuperación + BM25
docs_recuperados = hybrid_search(pregunta_prueba, top_k=15)

# 2. Reranking
docs_relevantes = rerank_documents(pregunta_prueba, docs_recuperados, top_n=3)

# 3. LLM Generation
respuesta_final, contexto_usado = generate_answer_with_citations(pregunta_prueba, docs_relevantes)
print(f"\n[RESPUESTA GENERADA]:\n{respuesta_final}")

# 4. Guardrails & Grounding
alucinacion, overlap = hallucination_guard(respuesta_final, contexto_usado)
grounding = evaluate_grounding(respuesta_final, contexto_usado)
print(f"\n[HALLUCINATION GUARD]: {'ALERTA: Posible Alucinación' if alucinacion else 'Pasa (Confiable)'} (Overlap Lexical: {overlap:.2f})")
print(f"[GROUNDING SCORE]: {grounding:.4f} (Escala 0-1)")

# 5. BLEU/ROUGE
rouge, bleu = evaluar_metricas(respuesta_final, respuesta_referencia_esperada)
print(f"\n[EVALUACIÓN AUTOMÁTICA]:")
print(f"- BLEU Score: {bleu:.4f}")
print(f"- ROUGE-1 F-measure: {rouge['rouge1'].fmeasure:.4f}")
print(f"- ROUGE-L F-measure: {rouge['rougeL'].fmeasure:.4f}")

--- QUERY: ¿Qué es el Poder Judicial? ---


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[RESPUESTA GENERADA]:
El Poder Judicial es uno de los tres poderes que, según la Constitución Política del Perú, conforman el Estado Peruano. El Poder Judicial está integrado por órganos jurisdiccionales jerárquicamente organizados y ejerce la potestad de administrar justicia en nombre de la Nación. [Doc 1]

[HALLUCINATION GUARD]: Pasa (Confiable) (Overlap Lexical: 1.00)
[GROUNDING SCORE]: 0.9798 (Escala 0-1)

[EVALUACIÓN AUTOMÁTICA]:
- BLEU Score: 0.0989
- ROUGE-1 F-measure: 0.3492
- ROUGE-L F-measure: 0.3175


13. Consultas Rápidas Demo

In [34]:
import warnings
warnings.filterwarnings("ignore")

print("SISTEMA RAG LEGAL - PERÚ")

pregunta_usuario = input("Escribe tu pregunta sobre leyes peruanas: ")

if pregunta_usuario.strip():
    # 1. Recuperación Híbrida (FAISS + BM25)
    docs_recuperados_demo = hybrid_search(pregunta_usuario, top_k=15)
    # 2. Reranking con Cross-Encoder
    docs_relevantes_demo = rerank_documents(pregunta_usuario, docs_recuperados_demo, top_n=3)
    # 3. Generación con LLM y Citation per sentence
    respuesta_limpia, contexto_demo = generate_answer_with_citations(pregunta_usuario, docs_relevantes_demo)
    print(f"\n[RESPUESTA]:\n{respuesta_limpia}")

else:
    print("\nNo ingresaste ninguna pregunta.")

SISTEMA RAG LEGAL - PERÚ
Escribe tu pregunta sobre leyes peruanas: Cuáles son los poderes del Perú


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[RESPUESTA]:
Los poderes del Perú son:

- El Poder Legislativo
- El Poder Ejecutivo
- El Poder Judicial

Esta clasificación se refleja en la Constitución Política del Perú, que define estos tres poderes como uno de los tres elementos constitutivos del Estado peruano.

[Doc 1]
